# Sesión 4B · Selenium: cuando la página se arma sola

En el notebook anterior sacamos una tabla de Wikipedia con `requests`. Eso
funciona cuando el dato **ya viene en el HTML**.

Hoy vamos por algo que no se deja: las ofertas de trabajo de
[bumeran.com.pe](https://www.bumeran.com.pe). Y de paso respondemos una
pregunta real:

> **¿Qué empleos se ofrecen hoy en el Perú, dónde, y cuántos son remotos?**

## 0. Comprobar el entorno

Esta celda no instala nada: **revisa** qué tienes y te dice qué falta.

Si falta algo, descomenta la última línea (la del `%pip`) y ejecútala. Se usa
`%pip` con `%`, no `!pip`: el `%pip` instala en **el mismo entorno donde corre
este notebook**, mientras que `!pip` puede instalar en otro Python y dejarte con
el mismo error de siempre.

Después de instalar, reinicia el kernel (*Kernel → Restart*).

In [1]:
# Revisa que esté todo. Si algo falta, te dice exactamente qué hacer.
import sys

faltan = []
for modulo, paquete in [("selenium", "selenium"), ("bs4", "beautifulsoup4"),
                        ("lxml", "lxml"), ("pandas", "pandas"),
                        ("requests", "requests")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)

print("Entorno:", sys.executable)

if faltan:
    print("\nFaltan:", " ".join(faltan))
    print("Descomenta la línea de abajo, ejecútala, y reinicia el kernel:")
    print(f"   %pip install {' '.join(faltan)}")
else:
    import selenium
    print("Todo instalado. selenium", selenium.__version__)

# %pip install selenium beautifulsoup4 lxml html5lib requests pandas

Entorno: /Users/alexanderquispe/Documents/GitHub/Diplomado_PUCP/.venv/bin/python
Todo instalado. selenium 4.49.0


In [2]:
!pip install --quiet selenium beautifulsoup4 lxml html5lib requests pandas

In [3]:

# Comprobar que quedo bien instalado y en que entorno estamos
import sys, selenium
print("Python   :", sys.executable)
print("selenium :", selenium.__version__)

Python   : /Users/alexanderquispe/Documents/GitHub/Diplomado_PUCP/.venv/bin/python
selenium : 4.49.0


In [4]:
import pandas as pd
import requests
import re
import time

## 1. Primero comprobamos que `requests` no alcanza

Esto no es un trámite: es el criterio para decidir qué herramienta usar.

In [5]:
headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

respuesta = requests.get("https://www.bumeran.com.pe/empleos.html",
                         headers=headers, timeout=30)
print("Código:", respuesta.status_code)
print("Tamaño del HTML:", len(respuesta.text), "caracteres")

Código: 200
Tamaño del HTML: 63242 caracteres


La página respondió bien. Ahora buscamos si adentro hay alguna oferta.

In [6]:
for palabra in ["Sueldo", "Publicado", "Remoto", "empresa"]:
    print(f"  '{palabra}': {respuesta.text.lower().count(palabra.lower())} veces")

  'Sueldo': 0 veces
  'Publicado': 0 veces
  'Remoto': 0 veces
  'empresa': 0 veces


**Cero.** El HTML llegó vacío de contenido: es solo el esqueleto. Las ofertas
las dibuja JavaScript **después**, en el navegador.

`requests` descarga el archivo pero no ejecuta JavaScript. Para esto hace
falta un navegador de verdad, y eso es **Selenium**.

## 2. Antes de scrapear: leer el `robots.txt`

Todo sitio publica en `/robots.txt` qué rutas pide que no se automaticen.
No es un candado técnico, es una norma — pero ignorarla es lo que hace que
te bloqueen, y en un trabajo profesional no se ignora.

In [7]:
robots = requests.get("https://www.bumeran.com.pe/robots.txt", headers=headers, timeout=30)
print(robots.text[:600])

User-agent: *
Disallow: /*recientes=true
Disallow: /*relevantes=true
Disallow: /empleos.html/111
Disallow: /empleos.html/100
Disallow: /empleos-area-*/111
Disallow: /empleos-area-*/100
Disallow: /en-*/empleos.html/111
Disallow: /en-*/empleos.html/100
Disallow: /empleos-busquedaext-
Disallow: /empleos/aptitus/*
Disallow: /*?localidades=*
Disallow: /*&localidades=*

Sitemap: https://www.bumeran.com.pe/sitemap_avisos_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_core_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_empresas_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_listados_ubi


Bumeran prohíbe rutas concretas (`?localidades=`, `?recientes=true`, ciertas
páginas), **pero no prohíbe el listado general** `/empleos.html`, que es el
que vamos a usar. Verificado antes de tocar nada.

## 3. Abrir un navegador desde Python

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Comprobación: ver el navegador con tus propios ojos

Corre esta celda una vez. Se va a **abrir una ventana de Chrome**, sola, que
navega a Bumeran y se cierra a los 8 segundos.

Si la ves, Selenium está funcionando. Si no ves nada pero tampoco da error,
también está funcionando: es el modo invisible, que es el que usaremos
después.

In [9]:
# Demostración: navegador VISIBLE (no usa la configuración de abajo)
demo = Options()
demo.add_argument("--window-size=1400,900")
# Ojo: aquí NO ponemos --headless, por eso se ve la ventana

chrome_demo = webdriver.Chrome(options=demo)
try:
    chrome_demo.get("https://www.bumeran.com.pe/empleos.html")
    time.sleep(8)                      # tiempo para mirarla
    print("Título:", chrome_demo.title)
finally:
    chrome_demo.quit()                 # se cierra sola
    print("Ventana cerrada")

Título: Empleos en Perú | Ofertas de Trabajo - Página 1 | Bumeran
Ventana cerrada


### Configuración

Aquí está el interruptor más importante del notebook:

| `HEADLESS` | Qué pasa |
|---|---|
| `True` | Chrome corre **sin ventana**. No ves nada, pero está trabajando. |
| `False` | Se abre una ventana y ves el navegador moverse solo. |

**Si pones `True` y no ves abrirse nada, eso es lo esperado.** No está fallando:
`--headless=new` ejecuta un Chrome completo que simplemente no dibuja la
pantalla. Es más rápido y no te roba el foco mientras trabajas.

Lo dejamos en `True` porque vamos a recorrer varias páginas. Para explorar un
sitio nuevo, ponlo en `False` y mira qué hace.

In [10]:
HEADLESS = True

def abrir_navegador():
    opciones = Options()
    if HEADLESS:
        opciones.add_argument("--headless=new")
    opciones.add_argument("--window-size=1920,1080")
    # Identificarse como un navegador normal
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opciones)


navegador = abrir_navegador()
print("Navegador abierto")

Navegador abierto


### Visitar la página

In [11]:
navegador.get("https://www.bumeran.com.pe/empleos.html")
navegador.title

''

**El título salió vacío.** No es un error: `.get()` devuelve el control apenas
el servidor responde, pero la página todavía se está armando. Si preguntas en
ese instante, no hay nada.

Esta es la causa del 90% de los scrapers de Selenium que "no encuentran nada".

## 4. Esperar a que cargue: el error número uno

Si buscas los datos apenas llegas, **no están todavía**. Hay dos formas de
esperar:

| Forma | Cómo funciona |
|---|---|
| `time.sleep(5)` | espera 5 segundos siempre, aunque ya esté listo |
| `WebDriverWait` | espera **hasta que aparezca** lo que le pides, y sigue |

La segunda es la correcta. La primera se usa como refuerzo cuando la página
sigue moviéndose después de cargar.

In [12]:
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/empleos/']"))
)
time.sleep(3)          # margen extra: la página sigue acomodándose
print("Contenido cargado")

Contenido cargado


### Comprobemos la diferencia

In [13]:
html_selenium = navegador.page_source
print("Con requests :", len(respuesta.text), "caracteres")
print("Con Selenium :", len(html_selenium), "caracteres")
print()
for palabra in ["Sueldo", "Publicado", "Remoto"]:
    print(f"  '{palabra}': {html_selenium.lower().count(palabra.lower())} veces")

Con requests : 63242 caracteres
Con Selenium : 285784 caracteres

  'Sueldo': 11 veces
  'Publicado': 19 veces
  'Remoto': 15 veces


Ahí está el contenido. Esa es toda la diferencia entre las dos herramientas.

## 5. Encontrar las ofertas

Lo natural sería buscar por clase CSS. Miremos qué clase tienen:

In [14]:
enlaces = navegador.find_elements(By.CSS_SELECTOR, "a[href]")
print("Enlaces totales en la página:", len(enlaces))

Enlaces totales en la página: 131


### El problema de las clases

Veamos la clase de una tarjeta de oferta:

In [15]:
ofertas = [a for a in enlaces
           if re.search(r"/empleos/.+-\d+\.html", a.get_attribute("href") or "")]

print("Ofertas encontradas:", len(ofertas))
print("Clase CSS de la primera:", ofertas[0].get_attribute("class"))

Ofertas encontradas: 20
Clase CSS de la primera: sc-ekQYnd bAylLX


`sc-ekQYnd bAylLX` no significa nada: es un nombre **generado automáticamente**
que cambia cada vez que la empresa actualiza su web. Un scraper que dependa
de esa clase se rompe la semana que viene.

Por eso usamos algo estable: **el patrón de la URL**. Toda oferta vive en
`/empleos/algo-NÚMERO.html`, y eso no va a cambiar.

```python
re.search(r"/empleos/.+-\d+\.html", href)
```

**Regla general:** prefiere URLs, `id` o atributos con significado. Las clases
generadas son el último recurso.

## 6. Mirar una oferta por dentro

In [16]:
primera = ofertas[0]
print(primera.get_attribute("href"))
print()
print(primera.text)

https://www.bumeran.com.pe/empleos/practicante-de-recursos-humanos-1118448397.html

Publicado ayer
PRACTICANTE DE RECURSOS HUMANOS
Confidencial
PRACTICANTE DE RRHH - Medio Tiempo c/s experienciaResumen del PuestoEstamos en la búsqueda de un Practicante de Recursos Humanos para unirse a nuestro equipo, en el CENTRO DE LIMA, a tiempo parcial. Buscamos personas proactivas, con gran disposición para aprender y desarrollarse en el área. Este rol es ideal para quienes desean iniciar su carrera profesional en gestión de talento humano.Responsabilidades Colaborar en la gestión de la publicidad de ofertas de empleo. Apoyar en el seguimiento y control de procesos internos del área. Manejo de controles y reportes. Coordinar la programación de entrevistas con candidatos. Brindar soporte general al equipo de Recursos Humanos en diversas tareas administrativas.Requisitos No se requiere experiencia previa, se valorará la actitud y las ganas de aprender. Habilidades de organización y gestión del tiemp

El texto viene en líneas, y siempre en el mismo orden. Los tres primeros
títulos son fecha, puesto y empresa:

In [17]:
titulos = primera.find_elements(By.CSS_SELECTOR, "h2, h3")
for i, t in enumerate(titulos[:3]):
    print(f"  [{i}] {t.text}")

  [0] Publicado ayer
  [1] PRACTICANTE DE RECURSOS HUMANOS
  [2] Confidencial


Y al final del texto están la ubicación y la modalidad:

In [18]:
lineas = [l.strip() for l in primera.text.split("\n") if l.strip()]
for i, l in enumerate(lineas):
    print(f"  [{i}] {l[:70]}")

  [0] Publicado ayer
  [1] PRACTICANTE DE RECURSOS HUMANOS
  [2] Confidencial
  [3] PRACTICANTE DE RRHH - Medio Tiempo c/s experienciaResumen del PuestoEs
  [4] Postulación rápida
  [5] Múltiples vacantes
  [6] Lince, Lima
  [7] Presencial


## 7. Una función que convierte una tarjeta en un diccionario

Aquí sí conviene una función: vamos a repetir esto 60 veces.

In [19]:
MODALIDADES = {"Presencial", "Remoto", "Híbrido"}

def leer_oferta(tarjeta):
    """Convierte una tarjeta de Bumeran en un diccionario."""
    lineas = [l.strip() for l in tarjeta.text.split("\n") if l.strip()]
    titulos = [t.text.strip() for t in tarjeta.find_elements(By.CSS_SELECTOR, "h2, h3")]

    # La modalidad, si aparece, es una de las últimas líneas
    modalidad = next((l for l in reversed(lineas) if l in MODALIDADES), None)

    # La ubicación es la línea justo antes de la modalidad
    ubicacion = None
    if modalidad and lineas.index(modalidad) > 0:
        ubicacion = lineas[lineas.index(modalidad) - 1]

    return {
        "publicado": titulos[0] if len(titulos) > 0 else None,
        "puesto":    titulos[1] if len(titulos) > 1 else None,
        "empresa":   titulos[2] if len(titulos) > 2 else None,
        "ubicacion": ubicacion,
        "modalidad": modalidad,
        "url":       tarjeta.get_attribute("href"),
    }


leer_oferta(primera)

{'publicado': 'Publicado ayer',
 'puesto': 'PRACTICANTE DE RECURSOS HUMANOS',
 'empresa': 'Confidencial',
 'ubicacion': 'Lince, Lima',
 'modalidad': 'Presencial',
 'url': 'https://www.bumeran.com.pe/empleos/practicante-de-recursos-humanos-1118448397.html'}

### Probar con las primeras cinco antes de lanzarse

In [20]:
pd.DataFrame([leer_oferta(o) for o in ofertas[:5]])

,publicado,puesto,empresa,ubicacion,modalidad,url
0,Publicado ayer,PRACTICANTE DE RECURSOS HUMANOS,Confidencial,"Lince, Lima",Presencial,https://www.bumeran.com.pe/empleos/practicante...
1,Publicado ayer,Asistente Contable Semi Senior,ESTUDIO SAAVEDRA TARMEÑO & ASOCIADOS SAC,"Lince, Lima",Presencial,https://www.bumeran.com.pe/empleos/asistente-c...
2,Publicado ayer,Trabaja Desde Casa/Remoto/Asesor Call Center W...,QUANTIA SOLUCIONES S.A.C.,"San Isidro, Lima",Remoto,https://www.bumeran.com.pe/empleos/trabaja-des...
3,Publicado ayer,430 Analista de Datos,INFORMATICA DELTA S.A.C.,"Lima, Lima",Híbrido,https://www.bumeran.com.pe/empleos/430-analist...
4,Publicado ayer,429 Ingeniero en Inteligencia Artificial,INFORMATICA DELTA S.A.C.,"Lima, Lima",Híbrido,https://www.bumeran.com.pe/empleos/429-ingenie...


## 8. Buscar un puesto concreto

Bumeran pone la búsqueda en la propia URL, así que no hace falta simular
escritura en el buscador:

```
/empleos-busqueda-analista-de-datos.html
```

In [21]:
def url_busqueda(termino, pagina=1):
    """Arma la URL de búsqueda de Bumeran."""
    slug = termino.lower().strip().replace(" ", "-")
    return f"https://www.bumeran.com.pe/empleos-busqueda-{slug}.html?page={pagina}"


url_busqueda("analista de datos")

'https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html?page=1'

In [22]:
navegador.get(url_busqueda("analista de datos"))
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/empleos/']"))
)
time.sleep(3)
navegador.title

'analista de datos: Empleos - Página 1 | Bumeran Perú'

## 9. El scraper completo

Junta todo: recorre varias páginas, espera en cada una, extrae y **hace una
pausa entre pedidos**.

Ese `time.sleep(2)` no es opcional. Sin él le caes al servidor con 10
consultas por segundo, y te bloquean con razón.

In [23]:
def scrapear(termino, paginas=3, espera=2):
    """Devuelve un DataFrame con las ofertas de Bumeran para un término."""
    navegador = abrir_navegador()
    recolectadas = []
    vistas = set()

    try:
        for pagina in range(1, paginas + 1):
            navegador.get(url_busqueda(termino, pagina))
            WebDriverWait(navegador, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/empleos/']"))
            )
            time.sleep(3)

            tarjetas = [a for a in navegador.find_elements(By.CSS_SELECTOR, "a[href]")
                        if re.search(r"/empleos/.+-\d+\.html", a.get_attribute("href") or "")]

            nuevas = 0
            for t in tarjetas:
                url = t.get_attribute("href")
                if url in vistas:            # evita duplicados entre páginas
                    continue
                vistas.add(url)
                recolectadas.append(leer_oferta(t))
                nuevas += 1

            print(f"  Página {pagina}: {nuevas} ofertas nuevas")
            time.sleep(espera)               # cortesía con el servidor

    finally:
        navegador.quit()                     # cerrar SIEMPRE, pase lo que pase

    return pd.DataFrame(recolectadas)

### Correrlo

El `try / finally` de arriba garantiza que el navegador se cierre aunque algo
falle. Sin eso se quedan procesos de Chrome abiertos comiéndose la memoria.

In [24]:
navegador.quit()      # cerramos el que teníamos abierto para explorar

empleos = scrapear("analista de datos", paginas=3)
empleos.shape

  Página 1: 20 ofertas nuevas


  Página 2: 20 ofertas nuevas


  Página 3: 20 ofertas nuevas


(60, 6)

In [25]:
empleos.head(10)

,publicado,puesto,empresa,ubicacion,modalidad,url
0,Publicado ayer,Analista de Datos,FRACTAL SOLUCIONES IT,"Lima, Lima",Híbrido,https://www.bumeran.com.pe/empleos/analista-de...
1,Publicado hace 5 días,Analista de Datos,Confidencial,"Cercado De Lima, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
2,Publicado ayer,430 Analista de Datos,INFORMATICA DELTA S.A.C.,"Lima, Lima",Híbrido,https://www.bumeran.com.pe/empleos/430-analist...
3,Publicado ayer,Analista de Explotación de Datos,Hitss Perú,"La Victoria, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
4,Actualizado hace 2 días,Analista de Datos,IBT GROUP,"San Isidro, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
5,Actualizado ayer,Analista de datos junior,Inetum Perú,"San Isidro, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
6,Publicado hace más de 15 días,Analista de Datos,IBT GROUP,"Callao, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
7,Publicado ayer,Analista de Gestión de Datos Maestros,CL SELECTION/DIVISION IT,"Lima, Lima",Presencial,https://www.bumeran.com.pe/empleos/analista-de...
8,Actualizado ayer,Analista de Datos Junior. (Hibrido),CSTI Corp,"Miraflores, Lima",Híbrido,https://www.bumeran.com.pe/empleos/analista-de...
9,Actualizado ayer,Analista de Datos Jr. (hibrido),CSTI Corp,"Miraflores, Lima",Híbrido,https://www.bumeran.com.pe/empleos/analista-de...


## 10. Guardar de inmediato

Los avisos de empleo cambian todos los días. Lo que scrapeaste hoy no lo
vuelves a conseguir mañana.

In [26]:
empleos.to_csv("empleos_bumeran.csv", index=False, encoding="utf-8")
print("Guardado:", empleos.shape[0], "ofertas")

Guardado: 60 ofertas


## 11. Y ahora es pandas otra vez

Desde aquí todo es lo de la sesión 3. Scrapear era solo conseguir la tabla.

In [27]:
empleos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   publicado  60 non-null     object
 1   puesto     60 non-null     object
 2   empresa    60 non-null     object
 3   ubicacion  60 non-null     object
 4   modalidad  60 non-null     object
 5   url        60 non-null     object
dtypes: object(6)
memory usage: 2.9+ KB


### ¿Cuántas ofertas son remotas?

In [28]:
empleos["modalidad"].value_counts(dropna=False)

modalidad
Presencial    44
Híbrido       15
Remoto         1
Name: count, dtype: int64

### ¿Dónde están los empleos?

In [29]:
empleos["ubicacion"].value_counts().head(10)

ubicacion
San Isidro, Lima            17
Miraflores, Lima             8
Lima, Lima                   7
Cercado De Lima, Lima        5
La Victoria, Lima            4
Surquillo, Lima              3
Arequipa, Arequipa           2
Jesús María, Lima            2
Cerro Colorado, Arequipa     2
El Agustino, Lima            1
Name: count, dtype: int64

### Separar distrito y ciudad

La ubicación viene como `"Lince, Lima"`. Se parte en dos columnas.

In [30]:
partes = empleos["ubicacion"].str.split(",", n=1, expand=True)
empleos["distrito"] = partes[0].str.strip()
empleos["ciudad"] = partes[1].str.strip() if partes.shape[1] > 1 else None
empleos[["puesto", "distrito", "ciudad", "modalidad"]].head()

,puesto,distrito,ciudad,modalidad
0,Analista de Datos,Lima,Lima,Híbrido
1,Analista de Datos,Cercado De Lima,Lima,Presencial
2,430 Analista de Datos,Lima,Lima,Híbrido
3,Analista de Explotación de Datos,La Victoria,Lima,Presencial
4,Analista de Datos,San Isidro,Lima,Presencial


### Las empresas que más publican

In [31]:
empleos["empresa"].value_counts().head(10)

empresa
CSTI Corp                           6
Confidencial                        5
INDRA PERU                          4
IBT GROUP                           2
Grupo Tawa                          2
ADECCO PERU S.A.                    2
UNIVERSIDAD TECNOLOGICA DEL PERU    2
Prosegur                            2
Grupo Lucky                         2
FRACTAL SOLUCIONES IT               1
Name: count, dtype: int64

### Modalidad por distrito

In [32]:
pd.crosstab(empleos["distrito"], empleos["modalidad"]).head(10)

modalidad,Híbrido,Presencial,Remoto
distrito,,,
Abancay,0,1,0
Arequipa,0,2,0
Barranco,1,0,0
Callao,0,1,0
Carabayllo,0,1,0
Cercado De Lima,1,4,0
Cerro Colorado,0,2,0
El Agustino,1,0,0
Jesús María,0,2,0


## 12. Comparar dos búsquedas

Ya con la función hecha, comparar carreras cuesta tres líneas.

In [33]:
contador = scrapear("contabilidad", paginas=2)
contador.shape

  Página 1: 20 ofertas nuevas


  Página 2: 20 ofertas nuevas


(40, 6)

In [34]:
COLUMNAS = ["publicado", "puesto", "empresa", "ubicacion", "modalidad", "url"]

comparacion = pd.concat([
    empleos[COLUMNAS].assign(busqueda="analista de datos"),
    contador[COLUMNAS].assign(busqueda="contabilidad"),
])

pd.crosstab(comparacion["busqueda"], comparacion["modalidad"], normalize="index").round(3) * 100

modalidad,Híbrido,Presencial,Remoto
busqueda,,,
analista de datos,25.0,73.3,1.7
contabilidad,27.5,70.0,2.5


In [35]:
comparacion.to_csv("empleos_comparacion.csv", index=False, encoding="utf-8")
comparacion.shape

(100, 7)

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Comprobar si hace falta Selenium | buscar el dato en `requests.get(url).text` |
| Consultar permisos | leer `/robots.txt` |
| Abrir navegador | `webdriver.Chrome(options=...)` |
| Ver sin ventana | `--headless=new` |
| Ir a una página | `.get(url)` |
| **Esperar bien** | `WebDriverWait(nav, 30).until(EC.presence_of_element_located(...))` |
| Buscar elementos | `.find_elements(By.CSS_SELECTOR, ...)` |
| Leer atributos | `.get_attribute("href")`, `.text` |
| Cerrar siempre | `try / finally` + `.quit()` |

### Las cuatro reglas

1. **Usa `requests` si puedes; Selenium solo si el dato no está en el HTML.**
   Selenium es diez veces más lento.
2. **No dependas de clases CSS generadas** (`sc-ekQYnd`). Usa patrones de URL,
   `id` o atributos con significado.
3. **Pausa entre pedidos.** Un `time.sleep(2)` es la diferencia entre ser un
   usuario intenso y ser un ataque.
4. **Guarda apenas extraes.** Los avisos de hoy no existen mañana.

### Si algo falla

| Error | Qué pasó |
|---|---|
| `TimeoutException` | el selector no apareció: cambió la página, o no cargó |
| `StaleElementReferenceException` | la página se volvió a dibujar; hay que buscar los elementos otra vez |
| `SessionNotCreatedException` | la versión de Chrome no coincide con el driver: actualiza Chrome |
| Lista vacía sin error | el selector ya no corresponde: ábrela con `HEADLESS = False` y mira |

---
## Para practicar

1. Corre `scrapear()` con el puesto de **tu** carrera.
2. ¿Qué distrito concentra más ofertas? ¿Cambia según la carrera?
3. Añade una columna con la fecha de hoy y guarda el CSV. Si repites el
   ejercicio cada semana, en un mes tienes una serie de tiempo del mercado
   laboral que nadie más tiene.